## Simulating two vessels passing a lock: two opposing vessels
OpenTNSim is a TU Delft library that enables the simulation of vessel agents over a network of waterways, including infrastructure and the environment. The library is based on the SimPy package for discrete-event simulation and uses the NetworkX package to create a network. Simulations are set up as follows:

0. Creating an environment  
1. Creating a network graph  
1+. Adding waterway infrastructure  
2. Adding vessel agents  
3. Running the simulation  
4. Inspecting the output   

In this notebook, we will simulate a lock complex with a lock chamber on a simple network. We model two opposing directed vessels that have to pass the complex. Hence, when one vessel is being levelled, the other vessel has to wait.

We start with importing the necessary libraries:

In [ ]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim

# import of basic core mixins and utils for creating objects, inspecting the output and plotting
from opentnsim.core import Movable, Identifiable, Locatable, Routable, Log, VesselProperties

from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.utils import create_object, inspect_object
from opentnsim.core.visualizations import generate_vessel_gantt_chart

# import of util for graph visualization
from opentnsim.graph.visualizations import plot_graph

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable
from opentnsim.lock.visualizations import spatially_visualize_lock_complex

# package(s) needed for inspecting the output
import pandas as pd

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

#### 0. Create environment
We create the SimPy environment, which includes a simulation start. We assign an epoch to the environment.

In [ ]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph
We create an undirected graph with two nodes (0 and 1), and a single edge of 10 kilometres between these nodes. The nodes and edges need to have a geometry in WGS84-coordinates. The edges should also be assigned a weight and a length in meters.

To be able to convert WGS84 degrees to meters, we make use of transformer functions:

In [ ]:
# define reference systems
wgs84eqd = pyproj.CRS('4087') #equidistant WGS84 in meters
wgs84rad = pyproj.CRS('4326') #radial WGS84 (used in GPS)

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

We create an empty base graph:

In [ ]:
# create an undirected graph
graph = nx.Graph()

And create the two nodes and add them to the graph:

In [ ]:
# Node 0
geometry_m = Point(-5000,0)
geometry = transform(wgs84eqd_to_wgs84rad, geometry_m)
graph.add_node('0', geometry=geometry)

# Node 1
geometry_m = Point(5000,0)
geometry = transform(wgs84eqd_to_wgs84rad, geometry_m)
graph.add_node('1', geometry=geometry)

And create the edge between these nodes, and add it to the graph:

In [ ]:
# add edges
geometry_m = LineString([Point(-5000, 0),Point(5000, 0)])
geometry = transform(wgs84eqd_to_wgs84rad, geometry_m)
graph.add_edge('0','1', geometry = geometry, weight=1, length_m=10000) #we need to have a geometry in WGS84 and a length in meters

We plot the graph

In [ ]:
plot_graph(graph)

As a last step, we add the graph to the environment:

In [ ]:
# add graph to environment
env.graph = graph

#### 1+ Adding infrastructure
The <strong>Lock Complex</strong> is an object that facilitates vessels to pass a hydraulic structure barrier that is traversable by means of a <strong>Lock Chamber</strong>. The complex has a <strong>Lock Master</strong> that schedules lock operations based on the traffic, and instructs vessels to wait in dedicated waiting areas or proceed to the lock chamber.

The lock complex in OpenTNSim consists of at least the following two objects:
1. A lock chamber;
2. Waiting areas on both sides of the lock chamber.

##### 1) Lock chamber
A Lock Chamber is schematized as a one-dimensional line object with a length capacity, a width and a depth. The chamber should be added to the environment, and a single edge (it cannot be placed on multiple edges), and at least have a name and distances from the lock gates to the edge nodes. More parameters can be added to the object and will be further discussed in the next notebooks. Each lock chamber has a <strong>Lock Operator</strong> that handles all the levelling operations.
<dl>
<dt>There are two ways of creating a lock chamber object:</dt>
<dd>a)   Using chamber dimensions, a selected edge and distances to nodes of the edge (<font color='red'>rule:</font> distances  + lock_length = edge_length);</dd>
<dd>b)   Using a geometry that overlaps the graph: chamber dimensions and distances to nodes of the edge are determined automatically</dd>
</dl>

In [ ]:
#We use approach a)
lock_chamber = IsLockChamber(env=env,
                             lock_length = 400, # in meters
                             lock_width = 50, # in meters
                             lock_depth = 10, # in meters
                             name='Lock',      
                             edge = ('0','1'),
                             levelling_time = 600, # in seconds
                             gate_opening_time = 60, # in seconds
                             gate_closing_time = 60, # in seconds
                             distance_from_start_node_to_lock_gate_A = 4800., # lock chamber will be located on the edge center (4.8 - 5.2 km)
                             distance_from_end_node_to_lock_gate_B = 4800., # lock chamber will be located on the edge center (4.8 - 5.2 km)
                             sailing_distance_to_crossing_point = 500.) # in meters

##### 2) Waiting areas
Waiting areas are considered as point objects with a certain integer capacity (default = infinity). The waiting areas should be assigned to the environment on an edge, and have a name and a distance to the start of the edge. Like the lock chamber, there are two ways of creating waiting areas object, but here we use a manual setup.

In [ ]:
waiting_area_A = IsLockWaitingArea(env=env,
                                   name = 'Waiting area A',
                                   edge = ('0','1'),
                                   orientation = 0, # orientation indicates that waiting area aligns with start node of lock edge (0) or not (1)
                                   distance_from_edge_start = 4300.) # waiting area A will be located at 500 m from the lock gate

waiting_area_B = IsLockWaitingArea(env=env,
                                   name = 'Waiting area B',
                                   edge = ('1','0'),
                                   orientation = 1, # orientation indicates that waiting area aligns with start node of lock edge (0) or not (1)
                                   distance_from_edge_start = 4300.) # waiting area B will be located at 500 m from the lock gate

We add the lock chambers and waiting areas to the lock complex object, and assign this to the environment. This object also needs a name, an environment, and registration nodes. At these nodes, the vessels will register themselves with the lock master. These nodes are the boundaries of the lock complex. All vessels that want to pass the lock complex should move over these nodes. Moreover, all the infrastructure of the lock complex should be within these nodes: a vessel should always pass a registration node, followed by a waiting area and a lock chamber.

In [ ]:
lock_complex = IsLockComplex(lock_chambers = [lock_chamber],
                             waiting_areas = [waiting_area_A, waiting_area_B],
                             registration_nodes = ['0','1'], # registration nodes is where the vessel registers to the lock planner
                             env=env, # add the lock complex to the environment
                             name = 'Lock complex',)

We can visualize the lock complex:

In [ ]:
spatially_visualize_lock_complex(lock_complex) # zooming in better shows the geometry of the lock, 
                                               # ignore the actual world location, as we consider a theoretical case

#### 2. Create agents
We will now create vessel agents, starting with the construction of the vessel agent object class.
A class is made up of building blocks that each add functionality. This approach improves the readability, organization, and modularity of code. Classes allow us to reuse earlier work and extend it easily.

We have to add the following classes to the Vessel class:
- **Identifiable**: assigns the vessel a unique ID to be traceable
- **Locatable**: allows the vessel to have a location
- **Routable**: allows the vessel to have a route over the network
- **Log**: allows the vessel to have a logbook to keep track of its actions at what time, which distance, and where (at what location)
- **VesselProperties**: allows the vessel to have properties, like vessel dimensions
- **Movable**: allows the object to move, with a fixed speed, while logging this activity
- **LockComplexTransverable**: allows to interact with a lock complex    

In [ ]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel", # name of the object
    (
        Identifiable,            # assigns the vessel a unique ID to be traceable
        Locatable,               # allows the vessel to have a location
        Routable,                # allows the vessel to have a route over the network
        Log,                     # allows the vessel to have a logbook to keep track of its actions at what time, which distance, and where (location)
        VesselProperties,        # allows the vessel to have properties, like vessel dimensions
        Movable,                 # allows the object to move, with a fixed speed, while logging this activity
        LockComplexTraversable,  # allows to interact with a lock complex           
    ), 
)

We define a mission for the vessels: to complete their route

In [ ]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

We can inspect what input is required to create a vessel and which parameters can be optionally added to this vessel:

In [ ]:
inspect_object(Vessel, show_parameter_table = True)

We create two vessels with opposing directions (using the above dataframe)

In [ ]:
# we create data dictionaries of the vessels
data_vessel_1 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 1",                                  # required by Identifiable
    "geometry": env.graph.nodes['0']['geometry'],        # required by Locatable
    "route": nx.dijkstra_path(env.graph, "0", "1"),      # required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 5,                                              # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:00:00')  # required by PassesLockComplex
}  

data_vessel_2 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 2",                                  # required by Identifiable
    "geometry": env.graph.nodes['1']['geometry'],        # required by Locatable
    "route": nx.dijkstra_path(env.graph, "1", "0"),      # required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 5,                                              # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:05:00')  # required by PassesLockComplex
}  

In [ ]:
# and construct the vessel
vessel_1 = Vessel(**data_vessel_1)
vessel_2 = Vessel(**data_vessel_2)

In [ ]:
# and add the vessels to the environment as processes:
env.process(mission(env, vessel_1))
env.process(mission(env, vessel_2));

#### 3. Run simulation
We run the simulation

In [ ]:
env.run()

#### 4. Inspect output
##### Vessel logbooks
We can inspect the dataframes of the two vessels. Messages emerge that correspond with the lock operation. For vessel 2, we also see waiting time, contributing to the delay to pass the lock.

In [ ]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel_1.logbook)

print("'{}' logbook data:".format(vessel_1.name))  
print('')

display(df)

**Question 1.**: Of which phases does the lock cycle consist from the perspective of the vessel?

1) Sailing to the waiting area
2) ...
3) ...
4) ...
5) ...
6) ...
7) ...
8) ...

##### Lock chamber logbook
The lock chamber also has a log which we can inspect. Messages consist of gate closing, converting (levelling), and gate opening. There are two operations.

In [ ]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_chamber.logbook)

print("'{}' logbook data:".format(lock_chamber.name))  
print('')

display(lock_df)

**Question 2**: Of which phases does the lock cycle consist from the perspective of the lock chamber?

1) Closing lock gates
2) ...
3) ...

##### Event chart
From the logbooks, we can create a Gantt chart. We see that the time to pass the edge with the lock chamber becomes variable. Moreover, we observe a cyclic behaviour of the lock chamber.

In [ ]:
df_eventtable = logbook2eventtable([vessel_1, vessel_2, lock_chamber])
generate_vessel_gantt_chart(df_eventtable)

**Question 3**: Explain: the duration of the following events:
1. "Sailing to Waiting Area"


2. "Sailing to first lock gate"


3. "Lock gate closing"


4. "Lock levelling"




##### Time-distance diagram
Lock chambers also have time-distance diagram plots, in which we can inspect the trajectories of the vessels and the lock chamber operation phases. We can now better understand why waiting time that emerged for the second vessel.

In [ ]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_chamber.plot(xlimmin = -5050, 
                        xlimmax = 5050, 
                        ylimmin = datetime.datetime(2025, 1, 1, 0, 0, 0),
                        ylimmax = datetime.datetime(2025, 1, 1, 1, 40, 0),
                        method='Plotly')

fig.update_layout(height=cm_to_pixels(20))

**Question 4**:

1. Explain: why the time-distance curves of vessels 1 and 2 are steeper in the lock, and what does it mean when the curve is vertical?

   
2. Explain: why vessel 2 waits, where it waits, and when it can sail into the lock?


3. Explain: the location at which both vessels wait in the lock?

